# 📊 09｜六城市氣象：同一份資料，依問題選圖並標清單位

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/09_weather_charts/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：同一份表格可以問出趨勢、比較、關係三種問題，**圖表由問題決定，不是由資料決定**；每張圖都要標出單位，單位不同的指標不放在同一個數值軸。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：確認資料完整與各欄單位，最後用**一張折線圖**回答「臺北一年中氣溫如何變化？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：把各指標整理成「城市 × 月份」的對照表，匯出成 Excel，表頭註明單位。
3. **選做挑戰（單元五～六）**：比較六城市的年降雨量，或觀察降雨量與日照時數的關係。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/weather_monthly.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。**一列代表：一個城市、一個月份的氣象平均值**，6 城市 × 12 個月＝72 列。數值為教學用近似值，不是中央氣象署觀測資料。

### 1. 欄位角色貼標籤
- 🔵 **條件 (Dimension)**：`城市`（類別）、`月份`（時間；雖然是 1～12 的數字，但不能加總或平均）。
- 🟢 **指標 (Metric)**：`平均氣溫`（°C）、`降雨量`（毫米）、`日照時數`（小時）、`降雨日數`（天）➔ **四個單位都不同**。
- ⚪ **標識 (ID)**：本例沒有流水號。

### 2. 五大提問積木與核心問題
- 📈 **趨勢（必做）**：臺北一年中平均氣溫如何變化？最熱、最冷是幾月？
- 📊 **比較（選做）**：六個城市中，哪個城市全年降雨量最多？
- 🔗 **關係（選做）**：降雨量多的月份，日照時數是否比較少？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **月份是時間，不是數值**：不能把月份加總或平均；圖上要依 1→12 排列，標成「1月…12月」。
- ⚠️ **不同單位不共用一個數值軸**：降雨量（毫米）和日照時數（小時）放在同一個 Y 軸，大小比較沒有意義。
- ⚠️ **能加總的才加總**：降雨量是每月累積值，可加總成年雨量；平均氣溫是平均值，加總 12 個月沒有意義。
- ⚠️ **相關不推因果**：雨多的月份日照少，只能說兩者同時發生；每個點是「某城市某月」，不是獨立的實驗。
- ⚠️ **教學用近似值**：不能拿來做真實的旅遊或防災決策。

### 4. 決定最適圖表
- **臺北 12 個月氣溫變化** ➔ **折線圖**（X 軸 1月→12月；Y 軸標示 °C，可依資料範圍設定，不必從 0）。
- **六城市全年降雨量（選做）** ➔ **垂直長條圖**（6 個類別比較總量，Y 軸從 0 開始，單位毫米）。
- **降雨量 vs 日照時數（選做）** ➔ **散佈圖**（兩個不同單位的指標各占一軸，點不連線）。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'weather_monthly.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/09_weather_charts/data/weather_monthly.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取資料，列出每個指標的單位（貼標籤）

> 🎯 **思維重點**：這是長格式：每個城市 12 列。先確認一列代表什麼，並替四個指標寫上單位——單位決定了能不能加總、能不能畫在同一軸。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/weather_monthly.csv：
1. 顯示前 5 筆資料、所有欄位名稱、總列數與各欄缺值數。
2. 列出有哪些城市、每個城市各有幾個月份。
3. 依資料卡，整理一張「欄位、角色、單位」的對照表（平均氣溫 °C、降雨量 毫米、日照時數 小時、降雨日數 天）。
4. 用文字說明：一列代表什麼？「月份」是指標還是條件？
先不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

df = pd.read_csv('data/weather_monthly.csv')
print("--- 前 5 筆資料 ---")
display(df.head())
print(f"\n總列數：{len(df)}；缺值：{df.isna().sum().sum()}")
print("各城市月份數：", df.groupby('城市')['月份'].count().to_dict())

units = pd.DataFrame({
    '欄位': ['城市', '月份', '平均氣溫', '降雨量', '日照時數', '降雨日數'],
    '角色': ['條件（類別）', '條件（時間）', '指標', '指標', '指標', '指標'],
    '單位': ['—', '月（1～12）', '°C', '毫米', '小時', '天'],
})
display(units)
print("➡️ 一列代表：一個城市、一個月份的氣象平均值；月份是時間順序，不是可加總的數值。")

### 📝 單元二【必做】：邊界檢核：確認完整性，先替每個問題選好圖

> 🎯 **思維重點**：畫圖前先做兩件事：①確認每個城市 1～12 月都齊全；②把「問題 → 需要的欄位 → 問題類型 → 圖表」寫成一張表，再請 AI 動手。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請先檢查資料，再整理選圖計畫：
1. 確認每個城市都剛好有 1～12 月各一列，沒有重複或缺漏。
2. 列出四個指標的最小值與最大值，檢查是否有不合理的數字（例如負的降雨量）。
3. 依下列三個問題，整理一張表：問題、需要的欄位與單位、問題類型（趨勢／比較／關係）、建議圖表、選圖理由：
   - 臺北一年中平均氣溫如何變化？
   - 六個城市中，哪個城市全年降雨量最多？
   - 降雨量多的月份，日照時數是否比較少？
先不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
complete = df.groupby('城市')['月份'].apply(lambda m: sorted(m) == list(range(1, 13)))
print("每個城市都有 1～12 月各一列：", complete.all())
print("重複的（城市, 月份）：", df.duplicated(['城市', '月份']).sum())
display(df[['平均氣溫', '降雨量', '日照時數', '降雨日數']].agg(['min', 'max']))

chart_plan = pd.DataFrame([
    ('臺北一年中平均氣溫如何變化？', '月份、平均氣溫（°C）', '趨勢', '折線圖', '時間由早到晚看走勢'),
    ('哪個城市全年降雨量最多？', '城市、降雨量（毫米，可加總）', '比較', '垂直長條圖', '6 個類別比總量，Y 軸從 0'),
    ('雨多的月份日照是否較少？', '降雨量（毫米）、日照時數（小時）', '關係', '散佈圖', '兩個數值各占一軸，點不連線'),
], columns=['問題', '需要的欄位與單位', '問題類型', '建議圖表', '選圖理由'])
display(chart_plan)

### 📝 單元三【必做】：提問積木【趨勢】— 臺北一年中氣溫如何變化？（折線圖）

> 🎯 **思維重點**：「時間 ＋ 指標」問變化，用【折線圖】；X 軸標成「1月…12月」，Y 軸標示 °C。氣溫的 0 不代表「沒有」，折線圖可依資料範圍設定 Y 軸。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請篩選「臺北」，畫出 12 個月平均氣溫的折線圖：
1. X 軸為月份，依 1→12 排序，刻度標成「1月、2月…12月」；Y 軸為平均氣溫（°C）。
2. 每個月加上圓點與數值標籤（保留一位小數）。
3. 用不同顏色標出最熱與最冷的月份。
4. 圖表標題、座標軸使用繁體中文，並在 Y 軸名稱寫出單位 °C。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

taipei = df[df['城市'] == '臺北'].sort_values('月份')
month_labels = [f'{m}月' for m in taipei['月份']]
hottest = taipei.loc[taipei['平均氣溫'].idxmax()]
coldest = taipei.loc[taipei['平均氣溫'].idxmin()]

plt.figure(figsize=(9, 5))
plt.plot(month_labels, taipei['平均氣溫'], marker='o', color='#3498db', linewidth=2)
for label, temp in zip(month_labels, taipei['平均氣溫']):
    plt.text(label, temp + 0.4, f'{temp:.1f}', ha='center', fontsize=9)
for point, color in [(hottest, '#e74c3c'), (coldest, '#2c3e50')]:
    plt.scatter(f"{point['月份']}月", point['平均氣溫'], color=color, s=90, zorder=3)
plt.ylim(taipei['平均氣溫'].min() - 3, taipei['平均氣溫'].max() + 3)
plt.title(f"臺北各月平均氣溫（最熱 {hottest['月份']}月、最冷 {coldest['月份']}月）", fontsize=14, fontweight='bold')
plt.xlabel('月份')
plt.ylabel('平均氣溫（°C）')
plt.grid(linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能說出一列代表「一城市 × 一月份」，並列出四個指標各自的單位。
- [ ] 能說出三種問題（趨勢、比較、關係）各該用什麼圖，以及為什麼。
- [ ] 能從折線圖指出臺北最熱與最冷的月份，並核對圖上數字與表格一致；X 軸標示為「1月…12月」。
- 💬 **討論題**：為什麼這張折線圖的 Y 軸不從 0 開始也沒關係，但長條圖一定要從 0 開始？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 資料 72 列，6 個城市各 12 個月，沒有缺漏；四個指標的單位分別是 °C、毫米、小時、天。
- 臺北折線圖顯示 **7 月最熱（29.7°C）**、**1～2 月最冷（16.8°C）**；X 軸為 1月→12月。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出「城市 × 月份」對照表，表頭註明單位

> 🎯 **思維重點**：把長格式轉成「列＝城市、欄＝月份」的對照表，方便閱讀；每個指標一個工作表，工作表名稱寫出單位。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請把四個指標各整理成「列＝城市、欄＝1月～12月」的對照表，並匯出 Excel：
1. 存為 output/六城市氣象對照表.xlsx（若 output 目錄不存在請自動建立）。
2. 每個指標一個工作表，工作表名稱寫出單位，例如「平均氣溫(°C)」「降雨量(毫米)」。
3. 降雨量、日照時數、降雨日數加上「全年合計」欄；平均氣溫不能加總，改加「全年平均」欄。
4. 匯出後重新讀取「降雨量(毫米)」工作表，核對全年合計與原始資料一致。
5. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
sheets = {'平均氣溫': '°C', '降雨量': '毫米', '日照時數': '小時', '降雨日數': '天'}
os.makedirs('output', exist_ok=True)
output_path = 'output/六城市氣象對照表.xlsx'
with pd.ExcelWriter(output_path) as writer:
    for metric, unit in sheets.items():
        table = df.pivot(index='城市', columns='月份', values=metric)
        table.columns = [f'{m}月' for m in table.columns]
        if metric == '平均氣溫':
            table['全年平均'] = table.mean(axis=1).round(1)   # 氣溫不能加總，只能平均
        else:
            table['全年合計'] = table.sum(axis=1).round(1)
        table.to_excel(writer, sheet_name=f'{metric}({unit})')

check = pd.read_excel(output_path, sheet_name='降雨量(毫米)', index_col=0)
print("重新讀取的全年降雨量：", check['全年合計'].to_dict())
print("原始資料計算：", df.groupby('城市')['降雨量'].sum().round(1).to_dict())

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 檔案！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較】— 哪個城市全年降雨量最多？（垂直長條圖）

> 🎯 **思維重點**：降雨量是每月累積值，可以加總成全年雨量；6 個城市比總量，用【垂直長條圖】，Y 軸從 0 開始。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請比較六個城市的全年降雨量：
1. 依城市加總 12 個月的降雨量，由多到少排序。
2. 畫出垂直長條圖：X 軸為城市，Y 軸為全年降雨量（毫米），從 0 開始，柱子上標註數值。
3. 全年雨量最多的城市用醒目顏色標示，寫在標題中。
4. 用一句話說明：為什麼降雨量可以加總，但平均氣溫不行？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
rain = df.groupby('城市')['降雨量'].sum().sort_values(ascending=False)
top_city = rain.idxmax()
colors = ['#e74c3c' if c == top_city else '#3498db' for c in rain.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(rain.index, rain.values, color=colors, width=0.55)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30, f'{bar.get_height():,.0f}', ha='center')
plt.ylim(0, rain.max() * 1.15)
plt.title(f'六城市全年降雨量（最多：{top_city}）', fontsize=13, fontweight='bold')
plt.xlabel('城市')
plt.ylabel('全年降雨量（毫米）')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()
print("💡 降雨量是每月累積的量，可以加總；平均氣溫是平均值，12 個月加總沒有意義。")

### 📝 單元六【選做挑戰】：提問積木【關係】— 雨多的月份，日照比較少嗎？（散佈圖）

> 🎯 **思維重點**：降雨量（毫米）與日照時數（小時）單位不同，不能畫在同一個 Y 軸；用【散佈圖】讓兩個指標各占一軸，點不連線。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請分析「降雨量」與「日照時數」是否有關聯：
1. 畫散佈圖：X 軸為降雨量（毫米）、Y 軸為日照時數（小時），每個點代表一個城市的一個月份，不要連線；不同城市用不同顏色並加圖例。
2. 計算兩者的皮爾森相關係數，顯示在標題中。
3. 用文字解讀關聯方向與強弱，並附註：相關不等於因果；每個點是「某城市某月」，同一城市的點彼此並不獨立。
4. 說明為什麼不建議把降雨量和日照時數畫成同一張圖的兩條折線、共用同一個 Y 軸。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
corr = df['降雨量'].corr(df['日照時數'])

plt.figure(figsize=(8, 5.5))
for city, part in df.groupby('城市'):
    plt.scatter(part['降雨量'], part['日照時數'], s=50, alpha=0.8, label=city)
plt.title(f'各城市各月降雨量與日照時數（r = {corr:.2f}）', fontsize=13, fontweight='bold')
plt.xlabel('月降雨量（毫米）')
plt.ylabel('月日照時數（小時）')
plt.legend(title='城市')
plt.grid(linestyle='--', alpha=0.4)
plt.show()
print(f"💡 相關係數 r = {corr:.2f}：雨多的月份，日照時數傾向較少。")
print("⚠️ 邊界提醒：相關不等於因果；兩者單位不同，不能共用同一個 Y 軸比大小。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：72 列資料完整；臺北折線圖 7 月最熱、1～2 月最冷，X 軸標成 1月→12月。
- [ ] **報表應用**：四個工作表名稱含單位，氣溫只算全年平均。
- [ ] **選做挑戰**：年降雨量用從 0 開始的長條圖；散佈圖只描述關聯，不推論因果。
- [ ] **AI 協作**：能說明自己提出的問題、AI 幫忙執行的事，以及自己如何確認結果。